# 1. Bureau Balance Processing Before Merging Into Bureau

## 1.1 Load Dataset

In [8]:
import pandas as pd

def load_data(data_path : str):
    df = pd.read_csv(f"{data_path}")

    return df

def save_data(df: pd.DataFrame, data_path : str):

    return df.to_csv(f"{data_path}", index = False)

df_bureau_balance = load_data("D:/Home Credit/Dataset/CSV/bureau_balance.csv")
df_bureau = load_data("D:/Home Credit/Dataset/CSV/bureau.csv")

In [9]:
bureau_balance = df_bureau_balance.copy()
bureau = df_bureau.copy()

## 1.2. Customized OneHotEncoder Function

In [10]:
def OHE(df : pd.DataFrame):
    # Get categorical columns
    categorical_columns = list(df.select_dtypes(include = ['object']))

    # OHE 
    df = pd.get_dummies(df, columns = categorical_columns, dummy_na = False, drop_first = False )

    # Return df
    return df 

## 1.3. Bureau Balance Processing

### OHE and Utlize aggregation to merge duplicated IDs into unique IDs

In [11]:
bureau_balance_copy = OHE(bureau_balance)

# Define aggregation dictionary
bureau_balance_aggregation = {}

# Get columns and its metrics to aggregate
bureau_balance_aggregation = {'MONTHS_BALANCE' : ['mean','max','min']}
for col in [cols for cols in bureau_balance_copy.drop(columns = ['SK_ID_BUREAU','MONTHS_BALANCE']).columns]:
    bureau_balance_aggregation[col] = ['mean']

# Aggregation
bureau_balance_agg = bureau_balance_copy.groupby("SK_ID_BUREAU").agg(bureau_balance_aggregation)

# Reset to normal format
bureau_balance_agg.columns = [f"{col}_{status}".upper() for col, status in bureau_balance_agg.columns]
bureau_balance_agg = bureau_balance_agg.reset_index()

C:\Users\Admin\AppData\Local\Temp\ipykernel_16060\312675149.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = list(df.select_dtypes(include = ['object']))


# 2. Merge Bureau Balance To Bureau

In [12]:
def merging(left_df : pd.DataFrame, right_df: pd.DataFrame, key : str, method : str):
    # Merging
    final_df = left_df.merge(right_df, left_on = key, right_on = key, how = method)

    # Return result
    return final_df

# Load Bureau Data
bureau_copy = OHE(bureau)

# Merging
final_bureau = merging(bureau_copy, bureau_balance_agg, key = 'SK_ID_BUREAU', method = 'left').drop(columns = ["SK_ID_BUREAU"])

C:\Users\Admin\AppData\Local\Temp\ipykernel_16060\312675149.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = list(df.select_dtypes(include = ['object']))


In [13]:
final_bureau.columns

Index(['SK_ID_CURR', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE',
       'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE',
       'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT',
       'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'DAYS_CREDIT_UPDATE',
       'AMT_ANNUITY', 'CREDIT_ACTIVE_Active', 'CREDIT_ACTIVE_Bad debt',
       'CREDIT_ACTIVE_Closed', 'CREDIT_ACTIVE_Sold',
       'CREDIT_CURRENCY_currency 1', 'CREDIT_CURRENCY_currency 2',
       'CREDIT_CURRENCY_currency 3', 'CREDIT_CURRENCY_currency 4',
       'CREDIT_TYPE_Another type of loan', 'CREDIT_TYPE_Car loan',
       'CREDIT_TYPE_Cash loan (non-earmarked)', 'CREDIT_TYPE_Consumer credit',
       'CREDIT_TYPE_Credit card', 'CREDIT_TYPE_Interbank credit',
       'CREDIT_TYPE_Loan for business development',
       'CREDIT_TYPE_Loan for purchase of shares (margin lending)',
       'CREDIT_TYPE_Loan for the purchase of equipment',
       'CREDIT_TYPE_Loan for working capital replenishment',
       '

# 3. Final Bureau Feature Engineering and Processing

### Feature Engineering

In [ ]:
# Ratios
final_bureau['DEBT_TO_CREDIT_RATIO'] = final_bureau['AMT_CREDIT_SUM_DEBT'] / (final_bureau['AMT_CREDIT_SUM'] + 0.5)
final_bureau['OVERDUE_TO_CREDIT_RATIO'] = final_bureau['AMT_CREDIT_SUM_OVERDUE'] / (final_bureau['AMT_CREDIT_SUM'] + 0.5)
final_bureau['OVERDUE_TO_DEBT_RATIO'] = final_bureau['AMT_CREDIT_SUM_OVERDUE'] / (final_bureau['AMT_CREDIT_SUM_DEBT'] + 0.5)
final_bureau['ANNUITY_TO_CREDIT_RATIO'] = final_bureau['AMT_ANNUITY'] / (final_bureau['AMT_CREDIT_SUM'] + 0.5)
final_bureau['ANNUITY_TO_DEBT_RATIO'] = final_bureau['AMT_ANNUITY'] / (final_bureau['AMT_CREDIT_SUM_DEBT'] + 0.5)


# Recent Features
recent_features = []
for month in [1,3,6]:
    recent = final_bureau.loc[final_bureau['DAYS_CREDIT'].between(-30 * month, 0)]
    agg = recent.groupby("SK_ID_CURR").agg(**{
        f"BUREAU_DAYS_CREDIT_MEAN_L{month}M" : ('DAYS_CREDIT', 'mean')})
    recent_features.append(agg)

recent_features = pd.concat(recent_features, axis = 1).reset_index()


# Feature Engineering
final_bureau['IS_ACTIVE'] = final_bureau['CREDIT_ACTIVE_Active'].astype(int)
final_bureau['IS_CLOSED'] = final_bureau['CREDIT_ACTIVE_Closed'].astype(int)

active_closed = []
for flag, name in [('IS_ACTIVE', "ACTIVE"), ('IS_CLOSED', "CLOSED")]:
    sub = final_bureau[final_bureau[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"BUREAU_{name}_COUNT" : ('DAYS_CREDIT', 'size'),
        f"BUREAU_{name}_DAYS_CREDIT_MAX" : ('DAYS_CREDIT', 'max'),
        f"BUREAU_{name}_DAYS_CREDIT_ENDDATE_MAX" : ('DAYS_CREDIT_ENDDATE', 'max'),
        f"BUREAU_{name}_AMT_CREDIT_SUM_SUM" : ('AMT_CREDIT_SUM', 'sum'),
        f"BUREAU_{name}_AMT_CREDIT_SUM_DEBT_SUM" : ('AMT_CREDIT_SUM_DEBT', 'sum'),
        f"BUREAU_{name}_AMT_CREDIT_SUM_OVERDUE_SUM" : ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
        f"BUREAU_{name}_AMT_ANNUITY_SUM" : ('AMT_ANNUITY', 'sum'),
        f"BUREAU_{name}_DEBT_TO_CREDIT_RATIO_MEAN" : ('DEBT_TO_CREDIT_RATIO', 'mean'),
        f"BUREAU_{name}_CREDIT_DAY_OVERDUE_MAX" : ('CREDIT_DAY_OVERDUE', 'max'),
    })
    active_closed.append(agg)

active_closed = pd.concat(active_closed, axis = 1).reset_index()

In [15]:
final_bureau.head()

,SK_ID_CURR,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,...,STATUS_5_MEAN,STATUS_C_MEAN,STATUS_X_MEAN,DEBT_TO_CREDIT_RATIO,OVERDUE_TO_CREDIT_RATIO,OVERDUE_TO_DEBT_RATIO,ANNUITY_TO_CREDIT_RATIO,ANNUITY_TO_DEBT_RATIO,IS_ACTIVE,IS_CLOSED
0,215354,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,...,NaN,NaN,NaN,0.000000,0.0,0.0,NaN,NaN,0,1
1,215354,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,...,NaN,NaN,NaN,0.761518,0.0,0.0,NaN,NaN,1,0
2,215354,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1,0
3,215354,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1,0
4,215354,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1,0


### Preprocessing

In [16]:
# Define aggregations dictionary
final_bureau_numeric_aggregations = {}
final_bureau_ohe_aggregations = {}

# Select features to do aggregation
final_bureau_numeric_aggregations = {
    'DAYS_CREDIT': ['min', 'max', 'mean'],
    'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
    'DAYS_CREDIT_UPDATE': ['min', 'max', 'mean'],
    'CREDIT_DAY_OVERDUE': ['min', 'max', 'mean'],
    'AMT_CREDIT_MAX_OVERDUE': ['mean'],
    'AMT_CREDIT_SUM': ['max','mean', 'min','sum'],
    'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_OVERDUE': ['mean'],
    'AMT_CREDIT_SUM_LIMIT': ['mean', 'max','min','sum'],
    'AMT_ANNUITY': ['max', 'mean', 'min','sum'],
    'CNT_CREDIT_PROLONG': ['sum'],
    'MONTHS_BALANCE_MEAN' : ['mean'],
    'MONTHS_BALANCE_MAX' : ['max'],
    'MONTHS_BALANCE_MIN' : ['min'],
    'STATUS_0_MEAN' : ['mean'],
    'STATUS_1_MEAN' : ['mean'],
    'STATUS_2_MEAN' : ['mean'],
    'STATUS_3_MEAN' : ['mean'],
    'STATUS_4_MEAN' : ['mean'],
    'STATUS_5_MEAN' : ['mean'],
    'STATUS_C_MEAN' : ['mean'],
    'STATUS_X_MEAN' : ['mean'],
    'DEBT_TO_CREDIT_RATIO' : ['mean', 'max'],
    'OVERDUE_TO_CREDIT_RATIO' : ['mean', 'max'],
    'OVERDUE_TO_DEBT_RATIO' : ['mean', 'max'],
    'ANNUITY_TO_CREDIT_RATIO' : ['mean', 'max'],
    'ANNUITY_TO_DEBT_RATIO': ['mean', 'max']
}

# OHE columns aggregation
ohe_cols = [col for col in final_bureau.columns if col.startswith(("CREDIT_ACTIVE_", "CREDIT_CURRENCY_", "CREDIT_TYPE_"))]
final_bureau_ohe_aggregations = {col : ['mean','sum'] for col in ohe_cols}


# Aggregation
final_bureau_agg = final_bureau.groupby('SK_ID_CURR').agg({**final_bureau_numeric_aggregations, **final_bureau_ohe_aggregations})
final_bureau_agg.columns = [f"BUREAU_{col}_{status}".upper() for col, status in final_bureau_agg.columns]
final_bureau_agg = final_bureau_agg.reset_index()
final_bureau_agg = final_bureau_agg.merge(recent_features, on = "SK_ID_CURR", how = 'left')
final_bureau_agg = final_bureau_agg.merge(active_closed, on = "SK_ID_CURR", how = 'left')
agg = final_bureau_agg

# Feature Engineering After Aggregation
agg['BUREAU_TOTAL_DEBT_TO_CREDIT'] = agg['BUREAU_AMT_CREDIT_SUM_DEBT_SUM'] / (agg['BUREAU_AMT_CREDIT_SUM_SUM'] + 1)
agg['BUREAU_ACTIVE_DEBT_TO_CREDIT'] = agg['BUREAU_ACTIVE_AMT_CREDIT_SUM_DEBT_SUM'] / (agg['BUREAU_ACTIVE_AMT_CREDIT_SUM_SUM'] + 1)
agg['BUREAU_ACTIVE_RATIO'] = agg['BUREAU_ACTIVE_COUNT'] / (agg['BUREAU_ACTIVE_COUNT'].fillna(0) + agg['BUREAU_CLOSED_COUNT'].fillna(0))
agg['BUREAU_LIMIT_USAGE'] = agg['BUREAU_AMT_CREDIT_SUM_DEBT_SUM'] / (agg['BUREAU_AMT_CREDIT_SUM_LIMIT_SUM'] + 1)

# 5. Save Data

In [17]:
save_data(agg, "D:/Home Credit/Dataset/CSV/bureau_agg_final.csv")